In [ ]:
!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q timm
!pip install -q segmentation-models-pytorch
!pip install -q albumentations
!pip install -q torchmetrics
!pip install -q einops

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

import albumentations as A

from tqdm import tqdm

In [ ]:
import rasterio
from rasterio.plot import reshape_as_image
from sklearn.model_selection import train_test_split

DATA_DIR = "/content/drive/MyDrive/ees/data"

# -----------------------------------------------------
# Load tif
# -----------------------------------------------------

def load_tif_as_array(filepath):

    with rasterio.open(filepath) as src:

        img = src.read().astype(np.float32)

        img = reshape_as_image(img)

    return img


# -----------------------------------------------------
# Normalize image
# -----------------------------------------------------

def normalize_image(img):

    img = img.astype(np.float32)

    img = img / np.max(img)

    return img


# -----------------------------------------------------
# Binarize mask
# -----------------------------------------------------

def binarize_mask(mask):

    mask = mask[:,:,0]

    if np.max(mask) <= 0.01:

        mask = (mask > 0.001).astype(np.uint8)

    elif np.max(mask) > 1:

        mask = (mask > 128).astype(np.uint8)

    else:

        mask = (mask > 0.5).astype(np.uint8)

    return mask


# -----------------------------------------------------
# Load 2024 image + mask
# -----------------------------------------------------

image = normalize_image(

    load_tif_as_array(

        os.path.join(
            DATA_DIR,
            "20241110_053942_45_24f7_3B_AnalyticMS_SR_8b_clip.tif"
        )

    )

)

mask = binarize_mask(

    load_tif_as_array(

        os.path.join(
            DATA_DIR,
            "20241110_053942_45_24f7_3B_AnalyticMS_SR_8b_clip_Hybrid_mask.tif"
        )

    )

)

print(image.shape)
print(mask.shape)

In [ ]:
# =====================================================
# PATCH EXTRACTION
# =====================================================

def extract_patches(
    image,
    mask,
    patch_size=128,
    stride=128
):

    X = []
    Y = []

    H,W,C = image.shape

    for i in range(0,H-patch_size+1,stride):

        for j in range(0,W-patch_size+1,stride):

            img_patch = image[
                i:i+patch_size,
                j:j+patch_size,
                :
            ]

            mask_patch = mask[
                i:i+patch_size,
                j:j+patch_size
            ]

            # Skip completely empty patches
            if np.sum(mask_patch)==0:

                continue

            X.append(img_patch)

            Y.append(mask_patch)

    X = np.array(X,dtype=np.float32)

    Y = np.array(Y,dtype=np.float32)

    Y = np.expand_dims(Y,-1)

    return X,Y


trainX,trainY = extract_patches(

    image,
    mask,
    patch_size=128,
    stride=128

)

print(trainX.shape)
print(trainY.shape)

In [ ]:
trainX,valX,trainY,valY = train_test_split(

    trainX,
    trainY,

    test_size=0.2,

    random_state=42,

    shuffle=True

)

print(trainX.shape)
print(valX.shape)

In [ ]:
mean = trainX.mean(axis=(0,1,2))
std = trainX.std(axis=(0,1,2))

print("Mean :", mean)
print("Std  :", std)

In [ ]:
class HRGLDDDataset(Dataset):

    def __init__(
        self,
        images,
        masks,
        mean,
        std,
        augment=False
    ):

        self.images = images
        self.masks = masks

        self.mean = mean
        self.std = std

        self.augment = augment

        if augment:

            self.transforms = A.Compose([

                A.HorizontalFlip(p=0.5),

                A.VerticalFlip(p=0.5),

                A.RandomRotate90(p=0.5),

                A.RandomBrightnessContrast(
                    p=0.5
                )

            ])

        else:

            self.transforms = None

    def __len__(self):
        return len(self.images)

    def __getitem__(self,idx):

        image = self.images[idx]
        mask = self.masks[idx]

        if self.transforms:

            transformed = self.transforms(
                image=image,
                mask=mask
            )

            image = transformed["image"]
            mask = transformed["mask"]

        image = (
            image - self.mean
        ) / self.std

        image = torch.tensor(
            image,
            dtype=torch.float32
        ).permute(2,0,1)

        mask = torch.tensor(
            mask,
            dtype=torch.float32
        ).permute(2,0,1)

        return image,mask

In [ ]:
train_dataset = HRGLDDDataset(
    trainX,
    trainY,
    mean,
    std,
    augment=True
)

val_dataset = HRGLDDDataset(
    valX,
    valY,
    mean,
    std,
    augment=False
)

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [ ]:
images,masks = next(iter(train_loader))

print(images.shape)
print(masks.shape)

In [ ]:
!pip install -U timm

In [ ]:
import timm

In [ ]:
class InputAdapter(nn.Module):

    def __init__(self):

        super().__init__()

        self.conv = nn.Conv2d(
            4,
            3,
            kernel_size=1
        )

    def forward(self,x):

        return self.conv(x)

In [ ]:
import timm
import torch
import torch.nn as nn


class DinoMultiEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.adapter = InputAdapter()

        self.backbone = timm.create_model(
            "vit_base_patch14_dinov2",
            pretrained=True,
            img_size=128
        )

    def forward(self,x):

        x = self.adapter(x)

        B = x.shape[0]

        x = self.backbone.patch_embed(x)

        cls_token = self.backbone.cls_token.expand(
            B,-1,-1
        )

        x = torch.cat(
            (cls_token,x),
            dim=1
        )

        x = x + self.backbone.pos_embed

        features = []

        for i,block in enumerate(
            self.backbone.blocks
        ):

            x = block(x)

            if i in [2,5,8,11]:

                features.append(x)

        return features

In [ ]:
encoder = DinoMultiEncoder()

print(
    encoder.backbone.pos_embed.shape
)

In [ ]:
device = "cuda"

encoder = DinoMultiEncoder().to(device)

images,masks = next(iter(train_loader))

images = images.to(device)

with torch.no_grad():

    feats = encoder(images)

for i,f in enumerate(feats):

    print(i,f.shape)

In [ ]:
import torch
import torch.nn as nn

In [ ]:
class TokenToMap(nn.Module):

    def __init__(self):

        super().__init__()

    def forward(self,x):

        x = x[:,1:,:]

        B,N,C = x.shape

        H = W = int(N**0.5)

        x = x.transpose(1,2)

        x = x.reshape(
            B,
            C,
            H,
            W
        )

        return x

In [ ]:
converter = TokenToMap()

for i,f in enumerate(feats):

    fmap = converter(f)

    print(i,fmap.shape)

In [ ]:
class FeatureCompression(nn.Module):

    def __init__(self):

        super().__init__()

        self.conv = nn.Sequential(

            nn.Conv2d(
                768,
                256,
                kernel_size=1
            ),

            nn.BatchNorm2d(256),

            nn.ReLU(inplace=True)

        )

    def forward(self,x):

        return self.conv(x)

In [ ]:
compressor = FeatureCompression().cuda()

for i,f in enumerate(feats):

    fmap = converter(f)

    fmap = compressor(
        fmap.cuda()
    )

    print(i,fmap.shape)

In [ ]:
class MultiLayerFusion(nn.Module):

    def __init__(self):

        super().__init__()

        self.compress = FeatureCompression()

        self.fusion = nn.Sequential(

            nn.Conv2d(
                1024,
                512,
                kernel_size=1
            ),

            nn.BatchNorm2d(512),

            nn.ReLU(inplace=True)

        )

    def forward(self,features):

        maps = []

        for f in features:

            fmap = TokenToMap()(f)

            fmap = self.compress(
                fmap
            )

            maps.append(fmap)

        fused = torch.cat(
            maps,
            dim=1
        )

        fused = self.fusion(
            fused
        )

        return fused

In [ ]:
fusion = MultiLayerFusion().cuda()

out = fusion(
    [f.cuda() for f in feats]
)

print(out.shape)

In [ ]:
for i,f in enumerate(feats):
    print(i,f.shape)

fused = fusion(
    [f.cuda() for f in feats]
)

print(fused.shape)

In [ ]:
class DoubleConv(nn.Module):

    def __init__(
        self,
        in_ch,
        out_ch
    ):

        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                in_ch,
                out_ch,
                3,
                padding=1
            ),

            nn.BatchNorm2d(
                out_ch
            ),

            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_ch,
                out_ch,
                3,
                padding=1
            ),

            nn.BatchNorm2d(
                out_ch
            ),

            nn.ReLU(inplace=True)

        )

    def forward(self,x):

        return self.block(x)

In [ ]:
class AttentionGate(nn.Module):

    def __init__(
        self,
        Fg,
        Fl,
        Fint
    ):

        super().__init__()

        self.Wg = nn.Sequential(

            nn.Conv2d(
                Fg,
                Fint,
                1
            ),

            nn.BatchNorm2d(
                Fint
            )

        )

        self.Wx = nn.Sequential(

            nn.Conv2d(
                Fl,
                Fint,
                1
            ),

            nn.BatchNorm2d(
                Fint
            )

        )

        self.psi = nn.Sequential(

            nn.Conv2d(
                Fint,
                1,
                1
            ),

            nn.BatchNorm2d(
                1
            ),

            nn.Sigmoid()

        )

        self.relu = nn.ReLU(inplace=True)

    def forward(
        self,
        g,
        x
    ):

        psi = self.relu(
            self.Wg(g) +
            self.Wx(x)
        )

        psi = self.psi(psi)

        return x * psi

In [ ]:
class DecoderBlock(nn.Module):

    def __init__(
        self,
        in_ch,
        out_ch
    ):

        super().__init__()

        self.up = nn.ConvTranspose2d(
            in_ch,
            out_ch,
            kernel_size=2,
            stride=2
        )

        self.conv = DoubleConv(
            out_ch,
            out_ch
        )

    def forward(self,x):

        x = self.up(x)

        x = self.conv(x)

        return x

In [ ]:
class DeepSupervisionHead(nn.Module):

    def __init__(
        self,
        in_ch
    ):

        super().__init__()

        self.head = nn.Conv2d(
            in_ch,
            1,
            kernel_size=1
        )

    def forward(self,x):

        return self.head(x)

In [ ]:
class ADSMSDecoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.dec1 = DecoderBlock(
            512,
            256
        )

        self.dec2 = DecoderBlock(
            256,
            128
        )

        self.dec3 = DecoderBlock(
            128,
            64
        )

        self.dec4 = DecoderBlock(
            64,
            32
        )

        self.ds1 = DeepSupervisionHead(
            256
        )

        self.ds2 = DeepSupervisionHead(
            128
        )

        self.ds3 = DeepSupervisionHead(
            64
        )

        self.ds4 = DeepSupervisionHead(
            32
        )

        self.final = nn.Conv2d(
            32,
            1,
            kernel_size=1
        )

    def forward(self,x):

        x1 = self.dec1(x)
        x2 = self.dec2(x1)
        x3 = self.dec3(x2)
        x4 = self.dec4(x3)

        out = self.final(x4)

        ds1 = self.ds1(x1)
        ds2 = self.ds2(x2)
        ds3 = self.ds3(x3)
        ds4 = self.ds4(x4)

        return (
            out,
            ds1,
            ds2,
            ds3,
            ds4
        )

In [ ]:
class DinoADSMSUNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder = DinoMultiEncoder()

        self.fusion = MultiLayerFusion()

        self.decoder = ADSMSDecoder()

    def forward(self,x):

        feats = self.encoder(x)

        fused = self.fusion(
            feats
        )

        out,ds1,ds2,ds3,ds4 = self.decoder(
            fused
        )

        out = nn.functional.interpolate(
            out,
            size=(128,128),
            mode='bilinear',
            align_corners=False
        )

        ds1 = nn.functional.interpolate(
            ds1,
            size=(128,128),
            mode='bilinear'
        )

        ds2 = nn.functional.interpolate(
            ds2,
            size=(128,128),
            mode='bilinear'
        )

        ds3 = nn.functional.interpolate(
            ds3,
            size=(128,128),
            mode='bilinear'
        )

        ds4 = nn.functional.interpolate(
            ds4,
            size=(128,128),
            mode='bilinear'
        )

        return (
            out,
            ds1,
            ds2,
            ds3,
            ds4
        )

In [ ]:
device = torch.device(
    "cuda"
)

model = DinoADSMSUNet().to(
    device
)

In [ ]:
images,masks = next(
    iter(train_loader)
)

images = images.to(device)

with torch.no_grad():

    outputs = model(images)

In [ ]:
for i,o in enumerate(outputs):

    print(i,o.shape)

In [ ]:
class DiceLoss(nn.Module):

    def __init__(self):
        super().__init__()

    def forward(self,preds,targets):

        preds = torch.sigmoid(preds)

        preds = preds.view(-1)
        targets = targets.view(-1)

        intersection = (preds * targets).sum()

        dice = (
            2. * intersection + 1e-6
        ) / (
            preds.sum() +
            targets.sum() +
            1e-6
        )

        return 1 - dice

In [ ]:
class BCEDiceLoss(nn.Module):

    def __init__(self):

        super().__init__()

        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()

    def forward(self,preds,targets):

        bce_loss = self.bce(
            preds,
            targets
        )

        dice_loss = self.dice(
            preds,
            targets
        )

        return (
            0.5*bce_loss +
            0.5*dice_loss
        )

In [ ]:
criterion = BCEDiceLoss()

def total_loss(outputs,targets):

    main,ds1,ds2,ds3,ds4 = outputs

    loss_main = criterion(
        main,
        targets
    )

    loss_ds1 = criterion(
        ds1,
        targets
    )

    loss_ds2 = criterion(
        ds2,
        targets
    )

    loss_ds3 = criterion(
        ds3,
        targets
    )

    loss_ds4 = criterion(
        ds4,
        targets
    )

    return (
        loss_main +
        0.4*loss_ds1 +
        0.3*loss_ds2 +
        0.2*loss_ds3 +
        0.1*loss_ds4
    )

In [ ]:
def iou_score(preds,targets):

    preds = torch.sigmoid(preds)

    preds = (preds > 0.5).float()

    intersection = (
        preds * targets
    ).sum()

    union = (
        preds +
        targets -
        preds*targets
    ).sum()

    return (
        intersection + 1e-6
    ) / (
        union + 1e-6
    )

In [ ]:
def dice_score(preds,targets):

    preds = torch.sigmoid(preds)

    preds = (preds > 0.5).float()

    intersection = (
        preds*targets
    ).sum()

    return (
        2*intersection + 1e-6
    ) / (
        preds.sum() +
        targets.sum() +
        1e-6
    )

In [ ]:
def precision_score(preds,targets):

    preds = torch.sigmoid(preds)

    preds = (preds > 0.5).float()

    TP = (
        preds*targets
    ).sum()

    FP = (
        preds*(1-targets)
    ).sum()

    return (
        TP + 1e-6
    ) / (
        TP + FP + 1e-6
    )

In [ ]:
def recall_score(preds,targets):

    preds = torch.sigmoid(preds)

    preds = (preds > 0.5).float()

    TP = (
        preds*targets
    ).sum()

    FN = (
        (1-preds)*targets
    ).sum()

    return (
        TP + 1e-6
    ) / (
        TP + FN + 1e-6
    )

In [ ]:
def f1_score(preds,targets):

    p = precision_score(
        preds,
        targets
    )

    r = recall_score(
        preds,
        targets
    )

    return (
        2*p*r
    ) / (
        p+r+1e-6
    )

In [ ]:
optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=1e-4,

    weight_decay=5e-5
)

In [ ]:
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(

    optimizer,

    T_max=50
)

In [ ]:
def train_epoch():

    model.train()

    running_loss = 0

    for images,masks in train_loader:

        images = images.to(device)

        masks = masks.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = total_loss(
            outputs,
            masks
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            0.7
        )

        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(train_loader)

In [ ]:
def validate_epoch():

    model.eval()

    val_loss = 0

    dice_total = 0
    iou_total = 0
    f1_total = 0

    with torch.no_grad():

        for images,masks in val_loader:

            images = images.to(device)

            masks = masks.to(device)

            outputs = model(images)

            loss = total_loss(
                outputs,
                masks
            )

            pred = outputs[0]

            val_loss += loss.item()

            dice_total += dice_score(
                pred,
                masks
            ).item()

            iou_total += iou_score(
                pred,
                masks
            ).item()

            f1_total += f1_score(
                pred,
                masks
            ).item()

    n = len(val_loader)

    return (
        val_loss/n,
        dice_total/n,
        iou_total/n,
        f1_total/n
    )

In [ ]:
best_dice = 0

history = []

for epoch in range(150):

    train_loss = train_epoch()

    val_loss,dice,iou,f1 = validate_epoch()

    scheduler.step()

    print(
        f"Epoch {epoch+1}/50 | "
        f"Train {train_loss:.4f} | "
        f"Val {val_loss:.4f} | "
        f"Dice {dice:.4f} | "
        f"IoU {iou:.4f} | "
        f"F1 {f1:.4f}"
    )

    history.append([
        train_loss,
        val_loss,
        dice,
        iou,
        f1
    ])

    if dice > best_dice:

        best_dice = dice

        torch.save(
            model.state_dict(),
            "/content/best_dino_adsms.pth"
        )

        print("Model Saved")

In [ ]:
model = DinoADSMSUNet().to(device)

model.load_state_dict(
    torch.load(
        "/content/best_dino_adsms.pth",
        map_location=device
    )
)

model.eval()

In [ ]:
import rasterio
from rasterio.plot import reshape_as_image

DATA_DIR="/content/drive/MyDrive/ees/data"

def load_tif(path):

    with rasterio.open(path) as src:

        img=src.read().astype(np.float32)

        img=reshape_as_image(img)

    return img


def normalize(img):

    return img/img.max()


def binarize(mask):

    mask=mask[:,:,0]

    if mask.max()>1:

        mask=(mask>128).astype(np.uint8)

    else:

        mask=(mask>0.5).astype(np.uint8)

    return mask


image2022=normalize(

    load_tif(
        DATA_DIR+
        "/20220503_050008_52_2474_3B_AnalyticMS_8b_clip.tif"
    )
)

mask2022=binarize(

    load_tif(
        DATA_DIR+
        "/20220503_050008_52_2474_3B_AnalyticMS_8b_clip_mask.tif"
    )
)

print(image2022.shape)
print(mask2022.shape)

In [ ]:
import math

def predict_large_image(
    model,
    image,
    mean,
    std,
    patch_size=128
):

    H,W,C=image.shape

    output=np.zeros((H,W),dtype=np.uint8)

    model.eval()

    with torch.no_grad():

        for y in range(0,H,patch_size):

            for x in range(0,W,patch_size):

                patch=image[
                    y:min(y+patch_size,H),
                    x:min(x+patch_size,W),
                    :
                ]

                ph=patch_size-patch.shape[0]
                pw=patch_size-patch.shape[1]

                patch=np.pad(

                    patch,

                    (
                        (0,ph),
                        (0,pw),
                        (0,0)
                    ),

                    mode="reflect"

                )

                patch=(patch-mean)/std

                patch=torch.tensor(
                    patch,
                    dtype=torch.float32
                ).permute(2,0,1)

                patch=patch.unsqueeze(0).to(device)

                pred=model(patch)[0]

                pred=torch.sigmoid(pred)

                pred=(pred>0.5).float()

                pred=pred.squeeze().cpu().numpy()

                output[
                    y:min(y+patch_size,H),
                    x:min(x+patch_size,W)
                ]=pred[
                    :patch.shape[1]-ph,
                    :patch.shape[2]-pw
                ]

    return output

In [ ]:
prediction=predict_large_image(

    model,

    image2022,

    mean,

    std,

    patch_size=128

)

In [ ]:
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

intersection=np.logical_and(
    prediction,
    
).sum()

union=np.logical_or(
    prediction,
   
).sum()

iou=intersection/(union+1e-8)

dice=2*intersection/(

    prediction.sum()

)

report=classification_report(

    mask2022.flatten(),

    prediction.flatten(),

    output_dict=True,

    zero_division=0

)

print("Accuracy :",report["accuracy"])

print("Precision :",report["1"]["precision"])

print("Recall :",report["1"]["recall"])

print("F1 :",report["1"]["f1-score"])

print("IoU :",iou)

print("Dice :",dice)

In [ ]:
rgb=image2022[:,:,:3]

plt.figure(figsize=(15,5))

plt.subplot(131)

plt.imshow(rgb)

plt.title("Satellite")

plt.axis("off")

plt.subplot(132)

plt.imshow(mask2022,cmap="gray")

plt.title("Ground Truth")

plt.axis("off")

plt.subplot(133)

plt.imshow(prediction,cmap="gray")

plt.title("DinoV2+ASDMS-UNet")

plt.axis("off")

plt.show()

In [ ]:
plt.figure(figsize=(8,8))

plt.imshow(rgb)

plt.imshow(

    prediction,

    cmap="Reds",

    alpha=0.4

)

plt.axis("off")

plt.title("Prediction Overlay")

plt.show()